# DE-03 — Transformation & Modeling

**Dataset:** `data/loan_data_03.csv`

This notebook covers profiling, standardization, null handling, deduplication, business and surrogate keys, referential integrity, fact grain, dimensions, SCD Type 1/2, accumulating snapshots, partition strategy, and semantic-ready serving tables.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_03.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

## Learning Content

### Transformation responsibilities

- **Profiling:** measure shape, nulls, uniqueness, ranges, and category distributions.
- **Standardization:** normalize names, text, types, units, and timestamps.
- **Null handling:** preserve unknowns, impute only with a documented business reason.
- **Deduplication:** use a declared business key and deterministic survivor rule.

### Modeling responsibilities

- A **business key** comes from the source; a **surrogate key** is controlled by the warehouse.
- A fact table must have one explicitly declared **grain**.
- Dimensions provide descriptive context; foreign keys enforce referential integrity.
- **SCD Type 1** overwrites history; **SCD Type 2** creates versioned rows.
- An **accumulating snapshot** updates milestones during a process lifecycle.
- Partitioning should follow access patterns and maintenance needs, not habit.

In [ ]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "null_count": raw.isna().sum(),
    "null_pct": (raw.isna().mean() * 100).round(2),
    "distinct_count": raw.nunique(dropna=True),
})
print(profile.to_string())
print("\nDuplicate Loan_ID values:", int(raw["Loan_ID"].duplicated().sum()))

In [ ]:
clean = raw.copy()

text_columns = ["Loan_ID", "Gender", "Married", "Dependents", "Education",
                "Self_Employed", "Property_Area", "Loan_Status"]
for column in text_columns:
    clean[column] = clean[column].astype("string").str.strip()

numeric_columns = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount",
                   "Loan_Amount_Term", "Credit_History"]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

clean = clean.drop_duplicates("Loan_ID", keep="last")
clean["Gender"] = clean["Gender"].fillna("Unknown")
clean["Self_Employed"] = clean["Self_Employed"].fillna("Unknown")
clean["TotalIncome"] = clean["ApplicantIncome"] + clean["CoapplicantIncome"]

print(clean.head(3).to_string(index=False))

## Hands-on / Demonstration

### Build customer and transaction dimensions/facts

For this training dataset, `Loan_ID` is the only stable row-level business key. Therefore, the applicant/customer dimension has one row per loan applicant. The fact grain is:

> **One row per loan application identified by Loan_ID.**

In [ ]:
# Applicant/customer dimension with a warehouse-controlled surrogate key.
dim_customer = (
    clean[["Loan_ID", "Gender", "Married", "Dependents", "Education", "Self_Employed"]]
    .drop_duplicates("Loan_ID")
    .sort_values("Loan_ID")
    .reset_index(drop=True)
)
dim_customer.insert(0, "Customer_Key", np.arange(1, len(dim_customer) + 1))

# Small conformed dimensions.
dim_property = pd.DataFrame({"Property_Area": sorted(clean["Property_Area"].dropna().unique())})
dim_property.insert(0, "Property_Key", np.arange(1, len(dim_property) + 1))

dim_status = pd.DataFrame({"Loan_Status": sorted(clean["Loan_Status"].dropna().unique())})
dim_status.insert(0, "Status_Key", np.arange(1, len(dim_status) + 1))
dim_status["Status_Label"] = dim_status["Loan_Status"].map({"Y": "Approved", "N": "Rejected"})

# Fact table at one row per loan application.
fact = (
    clean.merge(dim_customer[["Customer_Key", "Loan_ID"]], on="Loan_ID", validate="one_to_one")
    .merge(dim_property, on="Property_Area", validate="many_to_one")
    .merge(dim_status[["Status_Key", "Loan_Status"]], on="Loan_Status", validate="many_to_one")
    [["Loan_ID", "Customer_Key", "Property_Key", "Status_Key",
      "ApplicantIncome", "CoapplicantIncome", "TotalIncome",
      "LoanAmount", "Loan_Amount_Term", "Credit_History"]]
)

print("dim_customer:", dim_customer.shape)
print("dim_property:", dim_property.shape)
print("dim_status:", dim_status.shape)
print("fact:", fact.shape)

### Validate grain, keys, referential integrity, and totals

In [ ]:
source_loan_total = clean["LoanAmount"].sum(min_count=1)
fact_loan_total = fact["LoanAmount"].sum(min_count=1)

assert clean["Loan_ID"].is_unique
assert dim_customer["Customer_Key"].is_unique
assert fact["Loan_ID"].is_unique
assert fact["Customer_Key"].isin(dim_customer["Customer_Key"]).all()
assert fact["Property_Key"].isin(dim_property["Property_Key"]).all()
assert fact["Status_Key"].isin(dim_status["Status_Key"]).all()
assert np.isclose(source_loan_total, fact_loan_total, equal_nan=True)
assert len(clean) == len(fact)

print(f"Reconciled rows: {len(clean)}")
print(f"Reconciled loan amount: {fact_loan_total:,.2f}")

### SCD Type 1, SCD Type 2, and accumulating snapshot

The following small examples show the behavior of each pattern without modifying the original dataset.

In [ ]:
sample_customer = dim_customer.iloc[[0]].copy()

# SCD Type 1: overwrite the current attribute.
scd1 = sample_customer.copy()
scd1.loc[:, "Education"] = "Not Graduate"

# SCD Type 2: expire the old row and append a new version.
effective_at = pd.Timestamp("2026-02-01", tz="UTC")
old_version = sample_customer.assign(
    Valid_From=pd.Timestamp("2026-01-01", tz="UTC"),
    Valid_To=effective_at,
    Is_Current=False,
)
new_version = sample_customer.assign(
    Education="Not Graduate",
    Valid_From=effective_at,
    Valid_To=pd.NaT,
    Is_Current=True,
)
scd2 = pd.concat([old_version, new_version], ignore_index=True)

# Accumulating snapshot: milestone columns are updated as the application progresses.
accumulating_snapshot = pd.DataFrame([{
    "Loan_ID": clean.iloc[0]["Loan_ID"],
    "Applied_At": pd.Timestamp("2026-01-01", tz="UTC"),
    "Reviewed_At": pd.Timestamp("2026-01-02", tz="UTC"),
    "Decision_At": pd.Timestamp("2026-01-03", tz="UTC"),
    "Current_Status": "Decision completed",
}])

print("SCD Type 1 current education:", scd1.iloc[0]["Education"])
print("SCD Type 2 versions:", len(scd2))
print(accumulating_snapshot.to_string(index=False))

### Partition strategy and semantic-ready serving

For a growing loan fact table, a common PostgreSQL strategy is range partitioning by an immutable application date. Partition only when pruning, retention, or maintenance benefits justify the complexity.

The serving table below has a stable business meaning suitable for BI.

In [ ]:
semantic_approval_summary = (
    clean.groupby(["Property_Area", "Loan_Status"], dropna=False)
    .agg(
        Application_Count=("Loan_ID", "count"),
        Total_Loan_Amount=("LoanAmount", "sum"),
        Average_Total_Income=("TotalIncome", "mean"),
    )
    .reset_index()
)

assert semantic_approval_summary["Application_Count"].sum() == len(clean)
assert np.isclose(
    semantic_approval_summary["Total_Loan_Amount"].sum(),
    clean["LoanAmount"].sum(),
)
print(semantic_approval_summary.round(2).to_string(index=False))

## Enterprise Control

Declare grain before building transformations; reconcile records and amounts at every boundary.

The control must be executable, retained with a run identifier, and fail publication when a critical reconciliation is broken.

In [ ]:
model_control = {
    "declared_grain": "one row per Loan_ID",
    "source_rows": len(raw),
    "conformed_rows": len(clean),
    "fact_rows": len(fact),
    "source_loan_total": float(source_loan_total),
    "fact_loan_total": float(fact_loan_total),
    "referential_integrity": True,
}

assert model_control["conformed_rows"] == model_control["fact_rows"]
assert model_control["source_loan_total"] == model_control["fact_loan_total"]
print("DE-03 controls passed:", model_control)